# Tutorial: Understand the RF Xarray Structure

Use this notebook when you want to answer:
- what are the dimensions of the processed RF dataset?
- what are the named coordinates and xarray indexes?
- why is `cycle_id` a shared sparse axis across experiments?
- what changes when you select one experiment, one cycle, or one hostname?

The goal is to make the NetCDF feel like a physical measurement table rather than an abstract tensor.


In [ ]:
# Optional: uncomment when this Jupyter kernel misses the plotting/data dependencies.
# import sys
# !{sys.executable} -m pip install matplotlib numpy requests xarray pyyaml


In [ ]:
from pathlib import Path
import importlib.util
import sys

from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "csi_plot_utils.py").exists():
    NOTEBOOK_DIR = (NOTEBOOK_DIR / "processing").resolve()

UTILS_PATH = NOTEBOOK_DIR / "csi_plot_utils.py"
spec = importlib.util.spec_from_file_location("csi_plot_utils", UTILS_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load csi_plot_utils from {UTILS_PATH}")
csi = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = csi
spec.loader.exec_module(csi)


In [ ]:
EXPERIMENT_ID = "EXP003"
DATASET_PATH = None  # Set this to a specific .nc file when you do not want the newest match.
MAX_COORD_PREVIEW = 6
MAX_CYCLE_ROWS = 12


In [ ]:
ds, dataset_path = csi.open_dataset(experiment_id=EXPERIMENT_ID, dataset_path=DATASET_PATH)
available_cycles = csi.available_cycle_ids(ds, EXPERIMENT_ID)
if available_cycles.size == 0:
    raise ValueError(f"No CSI cycles available for experiment {EXPERIMENT_ID}")

SELECTED_CYCLE_ID = int(available_cycles[0])
experiment = ds.sel(experiment_id=EXPERIMENT_ID)
host_mask = experiment["csi_available"].sel(cycle_id=SELECTED_CYCLE_ID).values > 0
SELECTED_HOSTNAME = str(experiment["hostname"].values[host_mask][0])

print(f"Loaded dataset: {dataset_path}")
print(f"Selected experiment: {EXPERIMENT_ID}")
print(f"Example cycle for the walkthrough: {SELECTED_CYCLE_ID}")
print(f"Example hostname for the walkthrough: {SELECTED_HOSTNAME}")
csi.print_dataset_overview(ds)


## 1. The Named Axes, Coordinates, and Variables

The helper below summarizes the xarray layout directly from the dataset. This is the quickest way to see which names are dimensions, which names are coordinate indexes, and which names are real data variables.


In [ ]:
display(Markdown(csi.xarray_structure_markdown(ds, max_coord_preview=MAX_COORD_PREVIEW)))
ds.indexes


## 2. Make The Shared `cycle_id` Axis Tangible

This table shows one experiment slice as a per-cycle view. Each row is one rover stop. `csi_host_count` tells you how many hostnames reported CSI for that cycle. `position_valid` tells you whether the rover pose is usable for that cycle.

This is the key mental model: rover variables are per-cycle values, while CSI is a vector attached to that same cycle.


In [ ]:
cycle_table = csi.experiment_cycle_table(
    ds,
    EXPERIMENT_ID,
    max_rows=MAX_CYCLE_ROWS,
    only_cycles_with_csi=True,
)
print(f"Showing the first {cycle_table.sizes['cycle_id']} cycles with CSI for {EXPERIMENT_ID}")
cycle_table


## 3. What Changes When You Index Into The Dataset

xarray keeps the data readable because you select by coordinate names instead of integer positions. The walkthrough below shows how the remaining dimensions shrink as you move from the full dataset to one experiment, then one cycle, then one hostname.


In [ ]:
display(
    Markdown(
        csi.selection_walkthrough_markdown(
            ds,
            EXPERIMENT_ID,
            SELECTED_CYCLE_ID,
            SELECTED_HOSTNAME,
        )
    )
)


## 4. One Physical Measurement = One Rover Position + One CSI Vector

A physical RF measurement point is identified by the pair `(experiment_id, cycle_id)`. That pair gives you:
- one rover position
- one vector of CSI values across the active hostnames

The two helper calls below make that explicit.


In [ ]:
position = csi.cycle_position(ds, EXPERIMENT_ID, SELECTED_CYCLE_ID)
snapshot = csi.extract_csi_snapshot(ds, EXPERIMENT_ID, SELECTED_CYCLE_ID)

position


In [ ]:
print(
    f"CSI snapshot rows: {snapshot.sizes['hostname']} hostnames for "
    f"experiment {EXPERIMENT_ID}, cycle {SELECTED_CYCLE_ID}"
)
snapshot.isel(hostname=slice(0, min(8, snapshot.sizes['hostname'])))


## 5. Flatten The Sparse Dataset Into A Simple Measurement Table

The helper below converts the valid rover rows into a flat `measurement_index` table. This is often the easiest representation when you want to iterate over physical positions without thinking about the shared `cycle_id` axis.


In [ ]:
measurements = csi.positions_for_experiments(ds, [EXPERIMENT_ID])
print(f"Flattened valid measurement rows: {measurements.sizes['measurement_index']}")
measurements.isel(measurement_index=slice(0, min(10, measurements.sizes['measurement_index'])))


## Next Steps

After you are comfortable with the dataset layout, continue with:
- `plot_csi_positions.ipynb` for a short overview of trajectory and heatmaps
- `tutorial_rover_positions.ipynb` for the rover geometry and active antenna layout
- `tutorial_csi_per_position.ipynb` for extracting one CSI vector from one measurement point
